# 🛰️ SatQuery AI — Qwen2-VL-7B Fine-Tuning & Deployment Workflow

This reproducible notebook provides a complete workflow for fine-tuning `Qwen/Qwen2-VL-7B-Instruct` on Earth Observation Visual Question Answering (RSVQA / DOTA) datasets and deploying the FastAPI model server on Google Colab / T4 GPU.

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes peft datasets qwen-vl-utils fastapi uvicorn pyngrok python-multipart

## 1. Load Pretrained Qwen2-VL-7B-Instruct in 4-bit NF4 Quantization

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)
print("✅ Qwen2-VL-7B-Instruct loaded on T4 GPU!")

In [ ]:
import json
import re
from pathlib import Path
from qwen_vl_utils import process_vision_info

def ask_single_image(image_path, question, max_new_tokens=256):
    """Return a cautious, structured visual estimate from one image."""
    image_path = str(Path(image_path))
    if not Path(image_path).is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")

    guarded_question = (
        "Answer only from visible evidence in the image. "
        "Do not infer location, date, sensor metadata, hidden pixels, or facts outside the image. "
        "If the three classes cannot be distinguished reliably, use label 'uncertain'. "
        "Return JSON only with this exact shape: "
        '{"label":"agricultural|urban|forest|uncertain",'
        '"visible_evidence":["short observations"],"confidence":0.0}'
        " Confidence must be a number from 0.0 to 1.0 and is an uncalibrated visual estimate. "
        f"Question: {question}"
    )

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": guarded_question},
        ],
    }]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"\s*```$", "", cleaned, flags=re.IGNORECASE)

    json_match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if not json_match:
        raise ValueError(f"Model did not return valid JSON: {raw}")

    try:
        result = json.loads(json_match.group())
    except json.JSONDecodeError as exc:
        raise ValueError(f"Model did not return valid JSON: {raw}") from exc

    if not isinstance(result, dict):
        raise ValueError(f"JSON payload must be an object: {result!r}")

    allowed_labels = {"agricultural", "urban", "forest", "uncertain"}
    label = str(result.get("label", "uncertain")).strip().lower()
    evidence = result.get("visible_evidence", [])
    confidence = result.get("confidence", 0.0)

    if isinstance(confidence, str):
        try:
            confidence = float(confidence)
        except ValueError as exc:
            raise ValueError("confidence must be a number from 0.0 to 1.0") from exc

    if label not in allowed_labels:
        raise ValueError(f"Unsupported label returned by model: {label}")
    if not isinstance(evidence, list) or not all(isinstance(item, str) for item in evidence):
        raise ValueError("visible_evidence must be a list of strings")
    if not isinstance(confidence, (int, float)) or not 0.0 <= confidence <= 1.0:
        raise ValueError("confidence must be between 0.0 and 1.0")

    if label == "uncertain" or confidence < 0.55:
        label = "uncertain"

    return {
        "label": label,
        "visible_evidence": evidence[:5],
        "confidence": round(float(confidence), 2),
        "confidence_note": "Uncalibrated model estimate; verify with land-cover data before operational use.",
    }

def print_single_image_answer(result):
    print(f"Assessment: {result['label']}")
    print(f"Visible evidence: {'; '.join(result['visible_evidence']) or 'No reliable visual evidence returned.'}")
    print(f"Confidence: {result['confidence']:.0%} (uncalibrated model estimate)")
    print(result["confidence_note"])


In [ ]:
image_path = "2026-05-28-00_00_2026-05-28-23_59_Sentinel-2_L1C_True_color.jpg"
question = (
    "What is the dominant land use in this image: agricultural, urban, or forest? "
    "Use only visible evidence."
 )

result = ask_single_image(image_path, question)
print_single_image_answer(result)

## 2. Dataset Setup & QLoRA Fine-Tuning Configuration

- **Dataset Target**: RSVQA (Remote Sensing Visual Question Answering)
- **Input Features**: Multispectral satellite scene images + Natural Language VQA prompts
- **Evaluation Metrics**: VQA Accuracy, Precision, Recall, F1 Score, Bounding Box IoU

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

## 3. Deploy FastAPI Endpoint with Ngrok Tunnel

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from colab.app import app # Import FastAPI app from project repository

ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")
public_url = ngrok.connect(8000)
print(f"🚀 COLAB MODEL API DEPLOYED AT: {public_url}")

nest_asyncio.apply()
uvicorn.run(app, port=8000)